In [1]:
import numpy as np 

# XOR Dataset 
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]) 
y = np.array([[0], [1], [1], [0]]) 

class SimpleNN: 
    def __init__(self, learning_rate=0.1, activation='sigmoid', loss_type='mse'): 
        np.random.seed(42) 
        
        self.W1 = np.random.randn(2, 4) * 0.5 
        self.b1 = np.zeros((1, 4)) 
        self.W2 = np.random.randn(4, 1) * 0.5 
        self.b2 = np.zeros((1, 1)) 
        
        self.lr = learning_rate 
        self.activation_name = activation 
        self.loss_type = loss_type 
         
    def activation(self, x, derivative=False): 
        if self.activation_name == 'sigmoid': 
            return x * (1 - x) if derivative else 1 / (1 + np.exp(-x)) 
         
        elif self.activation_name == 'tanh': 
            return 1 - x**2 if derivative else np.tanh(x) 
         
        elif self.activation_name == 'relu': 
            return (x > 0).astype(float) if derivative else np.maximum(0, x) 
 
    def compute_loss(self, output, y): 
        if self.loss_type == 'mse': 
            return np.mean((output - y) ** 2) 
        elif self.loss_type == 'bce': 
            e = 1e-8 
            return -np.mean(y*np.log(output+e) + (1-y)*np.log(1-output+e)) 

    def loss_derivative(self, output, y): 
        if self.loss_type == 'mse': 
            return (output - y) 
        elif self.loss_type == 'bce': 
            e = 1e-8 
            return (output - y) / ((output+e)*(1-output+e)) 

    def forward(self, X): 
        self.z1 = np.dot(X, self.W1) + self.b1 
        self.a1 = self.activation(self.z1) 
        self.z2 = np.dot(self.a1, self.W2) + self.b2 
        self.a2 = self.activation(self.z2) 
        return self.a2 

    def backward(self, X, y, output): 
        m = X.shape[0] 
        
        dz2 = self.loss_derivative(output, y) * self.activation(self.a2, True) 
        dW2 = (1/m) * np.dot(self.a1.T, dz2) 
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True) 
        
        da1 = np.dot(dz2, self.W2.T) 
        dz1 = da1 * self.activation(self.a1, True) 
        dW1 = (1/m) * np.dot(X.T, dz1) 
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True) 
        
        self.W2 -= self.lr * dW2 
        self.b2 -= self.lr * db2 
        self.W1 -= self.lr * dW1 
        self.b1 -= self.lr * db1 

    def train(self, X, y, epochs=500): 
        for i in range(epochs): 
            output = self.forward(X) 
            self.backward(X, y, output) 

    def accuracy(self, X, y): 
        return np.mean((self.forward(X) > 0.5) == y) * 100 


# Learning rate comparison
for lr in [0.01, 0.1, 1.0]: 
    nn = SimpleNN(learning_rate=lr) 
    nn.train(X, y) 
    print(f"LR={lr}, Accuracy={nn.accuracy(X,y):.1f}%") 

# Loss function comparison
for loss in ['mse', 'bce']: 
    nn = SimpleNN(loss_type=loss) 
    nn.train(X, y) 
    print(f"Loss={loss}, Accuracy={nn.accuracy(X,y):.1f}%")

LR=0.01, Accuracy=50.0%
LR=0.1, Accuracy=75.0%
LR=1.0, Accuracy=50.0%
Loss=mse, Accuracy=75.0%
Loss=bce, Accuracy=50.0%
